# Assignment 3.1 — Feature Store: Neighborhood Feature Group

**Course:** AAI-540 · **Data:** `housing.csv` (California housing, 20,640 rows) + `housing_gmaps_data_raw.csv` (Google Maps reverse-geocode, 12,590 rows)

This notebook builds the **neighborhood feature group** for a house-price prediction tool, following the same SageMaker Feature Store patterns as Lab 3.1.

**How to use**
1. Upload this whole `homework-3-1` folder (notebook + both CSVs) to SageMaker Studio and run cells top to bottom.
2. Phase 1 (data loading + feature engineering) is plain pandas and can also be run locally for development.
3. Phase 2 needs the AWS Learner Lab (SageMaker execution role + default bucket).
4. Cells marked 📸 are the **screenshots required for submission** (query results for Brooktree, Fisherman's Wharf, Los Osos).

**Feature group schema** (all requirements from the assignment):

| feature | rule |
|---|---|
| `neighborhood` | primary key, from `neighborhood-political` |
| `event_time` | ingestion time, computed in Python |
| `lt_1h_ocean`, `inland`, `island`, `near_bay`, `near_ocean` | one-hot of `ocean_proximity`, averaged per neighborhood |
| `median_house_value` | mean per neighborhood, capped at 500,000 |
| `median_house_age` | mean per neighborhood, discretized into decade buckets (0-9, 10-19, …) |
| `total_households` | mean per neighborhood, rounded up to an integer |
| `bedrooms_per_household` | `total_bedrooms / households`; missing bedrooms imputed by postal-code average |

---
## Phase 1 — Data loading and feature engineering (no AWS needed)

### 1. Load the two datasets

`housing.csv` has one row per census block group; `housing_gmaps_data_raw.csv` has the Google Maps designation (including `neighborhood-political` and `postal_code`) keyed by the same `longitude,latitude` pair.

In [ ]:
import datetime
import math

import numpy as np
import pandas as pd

HOUSING_CSV = "housing.csv"
GMAPS_CSV = "housing_gmaps_data_raw.csv"

housing = pd.read_csv(HOUSING_CSV)
gmaps_raw = pd.read_csv(GMAPS_CSV)

print(f"housing: {housing.shape} | gmaps: {gmaps_raw.shape}")
print("ocean_proximity values:", housing["ocean_proximity"].unique().tolist())
print("missing total_bedrooms:", int(housing["total_bedrooms"].isna().sum()))
housing.head(3)

### 2. Join housing records to their Google Maps neighborhood

Left-join on `(longitude, latitude)`. Rows without a gmaps match have no `neighborhood-political` and **cannot enter the feature group** (the primary key may not be null) — they are counted and excluded below.

In [ ]:
gmaps = gmaps_raw[["longitude", "latitude", "neighborhood-political", "postal_code"]].copy()

joined = housing.merge(gmaps, on=["longitude", "latitude"], how="left")

matched = int(joined["neighborhood-political"].notna().sum())
print(f"join coverage: {matched:,} of {len(joined):,} housing rows "
      f"({matched / len(joined):.1%}) have a neighborhood")
print(f"distinct neighborhoods: {joined['neighborhood-political'].nunique()}")

### 3. Impute missing `total_bedrooms` by postal-code average

`housing.csv` has 207 rows with no `total_bedrooms`. Per the assignment, impute each missing value with the average of its postal code (all 207 rows have a postal code after the join). Postal codes that somehow had no observed bedrooms fall back to the global mean — reported so the choice is explicit.

In [ ]:
n_missing_before = int(joined["total_bedrooms"].isna().sum())

# postal_code stays numeric; pandas groupby.transform excludes NaN keys,
# so rows without a postal code keep NaN until the global fallback.
joined["total_bedrooms"] = joined["total_bedrooms"].fillna(
    joined.groupby("postal_code")["total_bedrooms"].transform("mean")
)
imputed_by_zip = n_missing_before - int(joined["total_bedrooms"].isna().sum())
joined["total_bedrooms"] = joined["total_bedrooms"].fillna(joined["total_bedrooms"].mean())

print(f"imputed by postal-code average: {imputed_by_zip} | by global fallback: "
      f"{n_missing_before - imputed_by_zip} | remaining missing: "
      f"{int(joined['total_bedrooms'].isna().sum())}")

### 4. Per-record features

- `bedrooms_per_household` = `total_bedrooms / households`
- one-hot encode `ocean_proximity` (5 categories). Feature Store feature names cannot contain spaces or `<`, so `<1H OCEAN` becomes `lt_1h_ocean`.
- cap `median_house_value` at 500,000 per record (the assignment cap; the raw data already tops out at 500,001).

In [ ]:
joined["bedrooms_per_household"] = joined["total_bedrooms"] / joined["households"]
joined["median_house_value_capped"] = joined["median_house_value"].clip(upper=500_000)

ocean_dummies = pd.get_dummies(joined["ocean_proximity"], dtype=float)
ocean_dummies.columns = [
    c.lower().replace("<", "lt_").replace(" ", "_") for c in ocean_dummies.columns
]
OCEAN_FEATURES = ["lt_1h_ocean", "inland", "island", "near_bay", "near_ocean"]
ocean_dummies = ocean_dummies[OCEAN_FEATURES]  # enforce all 5 columns, stable order
joined = pd.concat([joined, ocean_dummies], axis=1)
joined[["ocean_proximity"] + OCEAN_FEATURES + ["bedrooms_per_household"]].head(3)

### 5. Aggregate to the neighborhood feature group

Group by `neighborhood-political` and apply the assignment's aggregation rules. For the one-hot columns, the per-neighborhood mean is the fraction of the neighborhood's records in each ocean category (the aggregate form of a one-hot encoding).

In [ ]:
with_neighborhood = joined.dropna(subset=["neighborhood-political"]).copy()
print(f"rows entering the feature group: {len(with_neighborhood):,} "
      f"(excluded {len(joined) - len(with_neighborhood):,} rows without a neighborhood)")

grp = with_neighborhood.groupby("neighborhood-political")
neighborhood_fg = grp.agg(
    median_house_value=("median_house_value_capped", "mean"),
    median_house_age=("housing_median_age", "mean"),
    total_households=("households", "mean"),
    bedrooms_per_household=("bedrooms_per_household", "mean"),
    **{col: (col, "mean") for col in OCEAN_FEATURES},
).reset_index().rename(columns={"neighborhood-political": "neighborhood"})

# assignment rules: cap value, decade-bucket the age, round households UP to int
neighborhood_fg["median_house_value"] = neighborhood_fg["median_house_value"].clip(upper=500_000).round(2)
neighborhood_fg["median_house_age"] = neighborhood_fg["median_house_age"].apply(
    lambda a: f"{int(a // 10) * 10}-{int(a // 10) * 10 + 9}"
)
neighborhood_fg["total_households"] = neighborhood_fg["total_households"].apply(math.ceil).astype("int64")
neighborhood_fg["bedrooms_per_household"] = neighborhood_fg["bedrooms_per_household"].round(4)

# event_time: time of ingestion to the feature store, calculated in Python
neighborhood_fg["event_time"] = pd.Timestamp.now(tz="UTC")

COLUMN_ORDER = ["neighborhood", "event_time"] + OCEAN_FEATURES + [
    "median_house_value", "median_house_age", "total_households", "bedrooms_per_household",
]
neighborhood_fg = neighborhood_fg[COLUMN_ORDER]

print(f"feature group rows: {len(neighborhood_fg):,}")
neighborhood_fg.head(5)

### 6. Sanity checks before ingestion

In [ ]:
assert neighborhood_fg["neighborhood"].notna().all(), "primary key has nulls"
assert neighborhood_fg["neighborhood"].is_unique, "primary key not unique"
assert (neighborhood_fg["median_house_value"] <= 500_000).all(), "value cap violated"
assert neighborhood_fg["total_households"].dtype == np.int64, "households must be int"
assert np.allclose(neighborhood_fg[OCEAN_FEATURES].sum(axis=1), 1.0), "one-hot fractions must sum to 1"
print("All feature-group checks passed.")

QUERY_TARGETS = ["Brooktree", "Fisherman's Wharf", "Los Osos"]
preview = neighborhood_fg[neighborhood_fg["neighborhood"].isin(QUERY_TARGETS)]
assert len(preview) == 3, f"expected 3 query targets, found {len(preview)}"
print("\nExpected query results (what the Studio queries below should return):")
preview

---
## Phase 2 — SageMaker Feature Store (run in SageMaker Studio)

Setup mirrors Lab 3.1: boto3 session, SageMaker + FeatureStore runtime clients, execution role, and the default S3 bucket for the offline store.

In [ ]:
import boto3
import sagemaker

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = sagemaker.session.Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

role = sagemaker.get_execution_role()
default_s3_bucket_name = feature_store_session.default_bucket()
prefix = "sagemaker-featurestore-homework-3-1"
print(f"role: {role}\nbucket: s3://{default_s3_bucket_name}/{prefix}")

In [ ]:
import time

from sagemaker.feature_store.feature_group import FeatureGroup

FEATURE_GROUP_NAME = "neighborhood-feature-group"


def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")


neighborhood_feature_group = FeatureGroup(
    name=FEATURE_GROUP_NAME, sagemaker_session=feature_store_session
)

neighborhood_feature_group.create(
    s3_uri=f"s3://{default_s3_bucket_name}/{prefix}",
    record_identifier_name="neighborhood",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True,
)

wait_for_feature_group_creation_complete(feature_group=neighborhood_feature_group)

### Ingest the feature group (online + offline store)

In [ ]:
neighborhood_feature_group.ingest(data_frame=neighborhood_fg, max_workers=5, wait=True)
print(f"Ingested {len(neighborhood_fg):,} neighborhood records.")

### 📸 Query 1 (screenshot this) — online store point lookups

`get_record` for the three neighborhoods required by the assignment. These are the exact values to screenshot.

In [ ]:
for target in QUERY_TARGETS:
    record = featurestore_runtime.get_record(
        FeatureGroupName=FEATURE_GROUP_NAME,
        RecordIdentifierValueAsString=target,
    )
    features = {f["FeatureName"]: f["ValueAsString"] for f in record.get("Record", [])}
    print(f"\n=== {target} ===")
    for name in COLUMN_ORDER:
        print(f"  {name:>24}: {features.get(name)}")

### 📸 Query 2 (screenshot this) — offline store via Athena

Query the offline store with Athena, the same pattern as Lab 3.1 (`athena_query()`, `run()`, `wait()`, `as_dataframe()`). Note the doubled single-quote escape for the apostrophe in *Fisherman's Wharf*.

In [ ]:
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


in_clause = ", ".join(f"'{sql_escape(t)}'" for t in QUERY_TARGETS)

athena_query = neighborhood_feature_group.athena_query()
query_string = (
    f'SELECT {", ".join(COLUMN_ORDER)} '
    f'FROM "{athena_query.table_name}" '
    f"WHERE neighborhood IN ({in_clause})"
)
print(query_string)

output_location = f"s3://{default_s3_bucket_name}/{prefix}/query_results/"
athena_query.run(query_string=query_string, output_location=output_location)
athena_query.wait()
athena_query.as_dataframe()

---
## Submission checklist

1. 📸 Screenshot of **Query 1** (`get_record` for Brooktree, Fisherman's Wharf, Los Osos).
2. 📸 Screenshot of **Query 2** (Athena dataframe for the same three neighborhoods).
3. Download this notebook (File → Download → Notebook `.ipynb`) and upload it with the screenshots.

**Optional cleanup** (after grading): delete the feature group to avoid lingering charges —
`sagemaker_client.delete_feature_group(FeatureGroupName=FEATURE_GROUP_NAME)`.